# LeetCode #166 — Fraction to Recurring Decimal: chia tay + nhớ số dư đã gặp

**Mức độ:** Medium · **Chủ đề:** Hash Table, Math, String

Skeleton trên LeetCode:

```python
class Solution:
    def fractionToDecimal(self, numerator: int, denominator: int) -> str:
```

Mục tiêu của notebook:

1. Ôn **phép chia tay** (long division): mỗi bước nhân số dư với 10, lấy thương làm chữ số tiếp theo.
2. Nhận ra: phần thập phân **lặp lại** đúng khi một **số dư** xuất hiện lần thứ hai.
3. Dùng **bảng băm** `số dư → vị trí chữ số` để biết chỗ bắt đầu chu kỳ và chèn dấu ngoặc.
4. Xử lý **dấu** và các trường hợp biên (`0`, số âm, `−2^31`).

# 1. Bài toán là gì?

Cho hai số nguyên `numerator` (tử) và `denominator` (mẫu), trả về phân số dưới dạng **chuỗi số thập phân**. Nếu phần thập phân **lặp vô hạn**, đặt phần lặp trong ngoặc tròn.

Nếu có nhiều đáp án, trả về bất kỳ; đề đảm bảo chuỗi kết quả dài **dưới** `10^4` ký tự.

**Constraints (theo đề):**

- `-2^31 <= numerator, denominator <= 2^31 - 1`
- `denominator != 0`

---

## Ví dụ

```text
1 / 2     ->  "0.5"
2 / 1     ->  "2"
4 / 333   ->  "0.(012)"
1 / 6     ->  "0.1(6)"
-50 / 8   ->  "-6.25"
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Phép chia tay, phần dư và số thập phân tuần hoàn
2. Bảng băm (hash table) — `dict` và `set`

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Phép chia tay, phần dư và số thập phân tuần hoàn

### Chia tay (long division)

Chia `a` cho `b` (`a, b > 0`): phần nguyên `a // b`, phần dư `r = a % b`. Mỗi chữ số thập phân tiếp theo: **nhân dư với 10**, chia tiếp:

```text
1 / 7:   r=1 → 10//7 = 1, r=3 → 30//7 = 4, r=2 → 20//7 = 2, r=6 → 60//7 = 8,
         r=4 → 40//7 = 5, r=5 → 50//7 = 7, r=1  ← dư 1 đã gặp → lặp lại
         0.(142857)
```

In [1]:
def long_div_demo(a, b, max_digits=12):
    q, r = divmod(a, b)
    digits, rems = [], []
    while r and len(digits) < max_digits:
        rems.append(r)
        q2, r = divmod(r * 10, b)
        digits.append(q2)
    print(f"{a}/{b}: phần nguyên {q}, chữ số thập phân {digits}, các phần dư {rems}")

long_div_demo(1, 4)
long_div_demo(1, 7)
long_div_demo(1, 6)

1/4: phần nguyên 0, chữ số thập phân [2, 5], các phần dư [1, 2]
1/7: phần nguyên 0, chữ số thập phân [1, 4, 2, 8, 5, 7, 1, 4, 2, 8, 5, 7], các phần dư [1, 3, 2, 6, 4, 5, 1, 3, 2, 6, 4, 5]
1/6: phần nguyên 0, chữ số thập phân [1, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], các phần dư [1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]


### Vì sao tuần hoàn? — phần dư quyết định tất cả

Chữ số tiếp theo và phần dư tiếp theo **chỉ phụ thuộc phần dư hiện tại** (`r → (10r // b, 10r % b)`). Phần dư nằm trong `0 .. b−1`:

- Gặp dư `0` → chia hết, số thập phân **hữu hạn**.
- Không thì sau tối đa `b − 1` bước, một phần dư **lặp lại** (chuồng bồ câu) → từ đó mọi thứ lặp lại → **tuần hoàn**. Phần lặp bắt đầu ở vị trí lần đầu gặp phần dư đó.

Vậy để tìm chu kỳ: lưu **phần dư → vị trí chữ số** trong một `dict`.

In [2]:
def to_decimal_demo(a, b):
    q, r = divmod(a, b)
    digits, seen = [], {}
    while r and r not in seen:
        seen[r] = len(digits)
        q2, r = divmod(r * 10, b)
        digits.append(str(q2))
    if not r:
        return f"{q}" + ("." + "".join(digits) if digits else "")
    i = seen[r]
    return f"{q}." + "".join(digits[:i]) + "(" + "".join(digits[i:]) + ")"

for a_, b_ in [(1, 2), (2, 3), (1, 6), (22, 7), (1, 333), (4, 2)]:
    print(f"{a_}/{b_} = {to_decimal_demo(a_, b_)}")

1/2 = 0.5
2/3 = 0.(6)
1/6 = 0.1(6)
22/7 = 3.(142857)
1/333 = 0.(003)
4/2 = 2


### Lỗi hay gặp

- Lưu **chữ số** đã gặp thay vì **phần dư** → một chữ số lặp lại chưa chắc là bắt đầu chu kỳ: `3/14 = 0.2(142857)` có chữ số `2` xuất hiện lần hai ở vị trí thứ 4, nhưng chu kỳ bắt đầu từ chữ số `1`.
- Số âm: `divmod` của Python làm tròn **xuống** → xử lý dấu riêng, chia trên giá trị tuyệt đối.
- Dùng `float` → sai số, không tìm được chu kỳ dài.

## 2.2 Bảng băm (hash table) — `dict` và `set`

### Bảng băm là gì?

Bảng băm lưu các cặp **khóa → giá trị** sao cho **tra cứu, thêm, xóa theo khóa** chỉ mất trung bình `O(1)`.

Ý tưởng: một **hàm băm** biến khóa thành một con số; con số đó chỉ ra **ngăn** (bucket) trong một mảng nơi cất giá trị. Thay vì dò lần lượt cả danh sách (`O(n)`), ta tính hàm băm rồi nhảy thẳng tới ngăn.

```text
khóa "cam"  --hash--> 7  ──►  ngăn[7] = ("cam", 3)
khóa "táo"  --hash--> 2  ──►  ngăn[2] = ("táo", 5)
khóa "lê"   --hash--> 7  ──►  ngăn[7] đã có "cam" → VA CHẠM, Python tự xử lý (dò ngăn khác)
```

Python có sẵn hai kiểu dựa trên bảng băm:

| Kiểu | Lưu gì | Ví dụ |
|---|---|---|
| `dict` | khóa → giá trị | `{"cam": 3, "táo": 5}` |
| `set` | chỉ khóa (tập hợp, không trùng) | `{1, 4, 9}` |

**Khóa phải "hashable"** (bất biến): `int`, `str`, `tuple`, `frozenset` dùng được; `list`, `dict`, `set` thì **không**.

In [3]:
# --- dict: khóa -> giá trị ---
gia = {"cam": 3, "táo": 5}
gia["lê"] = 4                     # thêm
gia["cam"] = 6                    # cập nhật
print("gia['táo'] =", gia["táo"])
print("'xoài' có trong dict?", "xoài" in gia)        # kiểm tra khóa: O(1)
print("get có giá trị mặc định:", gia.get("xoài", 0))
del gia["lê"]                     # xóa
for k, v in gia.items():          # duyệt cặp khóa-giá trị
    print(" ", k, "->", v)

# --- set: tập hợp không trùng ---
s = {3, 1, 3, 2}
s.add(5); s.discard(1)
print("set:", s, "| 2 in s:", 2 in s)
print("giao:", {1, 2, 3} & {2, 3, 4}, "| hợp:", {1, 2} | {5}, "| hiệu:", {1, 2, 3} - {2})

gia['táo'] = 5
'xoài' có trong dict? False
get có giá trị mặc định: 0
  cam -> 6
  táo -> 5
set: {2, 3, 5} | 2 in s: True
giao: {2, 3} | hợp: {1, 2, 5} | hiệu: {1, 3}


### Vì sao nhanh? So sánh `list` và `set` khi kiểm tra phần tử

`x in list` phải so sánh lần lượt từng phần tử (`O(n)`); `x in set` chỉ tính hàm băm (`O(1)` trung bình).

In [4]:
import time as _time_hash

_data = list(range(200_000))
_as_set = set(_data)
_t0 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _data
_t1 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _as_set
_t2 = _time_hash.perf_counter()
print(f"100 lần 'in' trên list: {(_t1 - _t0) * 1000:.2f} ms | trên set: {(_t2 - _t1) * 1000:.3f} ms")

100 lần 'in' trên list: 718.36 ms | trên set: 0.111 ms


### Bốn mẫu dùng bảng băm hay gặp nhất

1. **Đếm tần suất** — `Counter` hoặc `d[x] = d.get(x, 0) + 1`.
2. **Nhóm theo khóa** — `defaultdict(list)`, khóa là "đặc trưng chung" của nhóm.
3. **Ghi nhớ thứ đã thấy** — lưu giá trị → vị trí để trả lời "đã gặp phần bù chưa?" trong `O(1)` (kiểu Two Sum).
4. **Đánh dấu đã thăm** — `visited = set()` khi duyệt đồ thị / lưới.

In [5]:
from collections import Counter, defaultdict

# 1) Đếm
print("Counter:", Counter("banana"))

# 2) Nhóm theo khóa: nhóm các từ theo chữ cái đầu
nhom = defaultdict(list)
for w in ["an", "bò", "ba", "ăn", "ai"]:
    nhom[w[0]].append(w)
print("Nhóm:", dict(nhom))

# 3) Ghi nhớ đã thấy: tìm hai chỉ số có tổng = target trong một lượt
def hai_so_tong(nums, target):
    vi_tri = {}                       # giá trị -> chỉ số
    for i, x in enumerate(nums):
        if target - x in vi_tri:      # phần bù đã gặp trước đó?
            return vi_tri[target - x], i
        vi_tri[x] = i
    return None
print("hai_so_tong([2, 7, 11, 15], 9) =", hai_so_tong([2, 7, 11, 15], 9))

# 4) Đã thăm
da_tham = set()
for v in [3, 1, 3, 2, 1]:
    if v in da_tham:
        print("gặp lại", v)
    da_tham.add(v)

Counter: Counter({'a': 3, 'n': 2, 'b': 1})
Nhóm: {'a': ['an', 'ai'], 'b': ['bò', 'ba'], 'ă': ['ăn']}
hai_so_tong([2, 7, 11, 15], 9) = (0, 1)
gặp lại 3
gặp lại 1


### Độ phức tạp và lưu ý

| Thao tác | `dict` / `set` (trung bình) | `list` |
|---|---|---|
| kiểm tra `x in ...` | `O(1)` | `O(n)` |
| thêm / xóa theo khóa | `O(1)` | xóa giữa: `O(n)` |
| duyệt toàn bộ | `O(n)` | `O(n)` |

Lỗi hay gặp:

- Dùng `list` làm khóa → `TypeError: unhashable type`. Đổi sang `tuple(...)`.
- Thêm/xóa khóa **trong lúc** đang duyệt chính dict đó → `RuntimeError`.
- Truy cập `d[k]` khi `k` chưa có → `KeyError`; dùng `d.get(k, mặc_định)` hoặc `defaultdict`.

In [6]:
try:
    {[1, 2]: "x"}
except TypeError as e:
    print("Khóa là list:", e)
print("Khóa là tuple thì được:", {(1, 2): "x"})

Khóa là list: unhashable type: 'list'
Khóa là tuple thì được: {(1, 2): 'x'}


## 2.3 Áp dụng vào bài này

Đây chính là **phép chia tay**: phần nguyên, rồi mỗi chữ số thập phân là `(dư · 10) // mẫu`. Chu kỳ xuất hiện đúng khi một **phần dư** lặp lại, nên ta lưu `dư → vị trí chữ số` trong **bảng băm** để biết chỗ đặt dấu `(`. Dấu xử lý riêng, chia trên giá trị tuyệt đối.

# 3. Nền tảng

In [7]:
from typing import List
import random
from fractions import Fraction

q, r = divmod(22, 7)
print("22 / 7: phần nguyên", q, "dư", r)
print("divmod với số âm:", divmod(-7, 2), "← Python làm tròn XUỐNG, nên xử lý dấu riêng rồi dùng abs")
print("Fraction(4, 333) =", Fraction(4, 333), "| float:", 4 / 333)

22 / 7: phần nguyên 3 dư 1
divmod với số âm: (-4, 1) ← Python làm tròn XUỐNG, nên xử lý dấu riêng rồi dùng abs
Fraction(4, 333) = 4/333 | float: 0.012012012012012012


`float` chỉ có ~16 chữ số có nghĩa nên **không** thể dùng để tìm chu kỳ lặp (có thể dài hàng nghìn chữ số). Ta tự làm phép chia tay bằng số nguyên.

## Phép chia tay

Chia `4` cho `333`:

```text
số dư  ×10    chia 333    chữ số   số dư mới
  4  →  40  →  40 // 333 =  0     40
 40  → 400  → 400 // 333 =  1     67
 67  → 670  → 670 // 333 =  2      4   ← số dư 4 đã gặp ở bước đầu!
  4  →  40  → … lặp lại y hệt từ đây
```

Mỗi chữ số tiếp theo **chỉ phụ thuộc vào số dư hiện tại**. Số dư chỉ có thể là `0, 1, …, d − 1`, nên sau tối đa `d` bước, hoặc số dư bằng `0` (hết, thập phân hữu hạn), hoặc một số dư **lặp lại** → mọi chữ số từ đó lặp theo chu kỳ.

In [8]:
def long_division_steps(a, d, k=8):
    r = a % d
    for _ in range(k):
        r10 = r * 10
        print(f"   dư {r:>3} → ×10 = {r10:>4} → chữ số {r10 // d}, dư mới {r10 % d}")
        r = r10 % d
        if r == 0:
            break

long_division_steps(4, 333, 4)

   dư   4 → ×10 =   40 → chữ số 0, dư mới 40
   dư  40 → ×10 =  400 → chữ số 1, dư mới 67
   dư  67 → ×10 =  670 → chữ số 2, dư mới 4
   dư   4 → ×10 =   40 → chữ số 0, dư mới 40


# 4. Ý tưởng chính

```text
1) Dấu: âm nếu đúng một trong hai số âm (và tử khác 0). Làm việc với |tử|, |mẫu|.
2) Phần nguyên: q, r = divmod(|tử|, |mẫu|). Nếu r == 0 → xong.
3) Phần thập phân: seen = {}  (số dư → vị trí chữ số trong danh sách digits)
   while r != 0:
       if r in seen: chèn "(" tại seen[r], thêm ")" ở cuối; dừng
       seen[r] = len(digits)
       r *= 10; digits.append(r // |mẫu|); r %= |mẫu|
```

Vì sao lưu **vị trí** của số dư? Chữ số sinh ra **từ** số dư `r` bắt đầu tại vị trí `seen[r]`; khi gặp lại `r`, đoạn từ `seen[r]` tới hiện tại chính là chu kỳ.

In [9]:
def fraction_to_decimal(numerator: int, denominator: int) -> str:
    if numerator == 0:
        return "0"
    sign = "-" if (numerator < 0) != (denominator < 0) else ""
    n, d = abs(numerator), abs(denominator)
    q, r = divmod(n, d)
    if r == 0:
        return sign + str(q)
    digits, seen = [], {}
    while r != 0:
        if r in seen:
            i = seen[r]
            return f"{sign}{q}." + "".join(digits[:i]) + "(" + "".join(digits[i:]) + ")"
        seen[r] = len(digits)
        r *= 10
        digits.append(str(r // d))
        r %= d
    return f"{sign}{q}." + "".join(digits)

for a, b in [(1, 2), (2, 1), (4, 333), (1, 6), (-50, 8), (1, 7), (0, -5)]:
    print(f"{a:>4} / {b:<4} = {fraction_to_decimal(a, b)}")

   1 / 2    = 0.5
   2 / 1    = 2
   4 / 333  = 0.(012)
   1 / 6    = 0.1(6)
 -50 / 8    = -6.25
   1 / 7    = 0.(142857)
   0 / -5   = 0


# 5. Mô phỏng

In [10]:
def trace_fraction(numerator, denominator):
    n, d = abs(numerator), abs(denominator)
    q, r = divmod(n, d)
    print(f"phần nguyên {q}, dư {r}")
    digits, seen = [], {}
    while r != 0:
        if r in seen:
            print(f"   gặp lại dư {r} (lần đầu ở vị trí {seen[r]}) → chu kỳ là {''.join(digits[seen[r]:])}")
            break
        seen[r] = len(digits)
        r *= 10
        digits.append(str(r // d))
        r %= d
        print(f"   vị trí {len(digits) - 1}: chữ số {digits[-1]}, dư mới {r:<3} | seen = {seen}")
    print("→", fraction_to_decimal(numerator, denominator))

trace_fraction(1, 6)
trace_fraction(22, 7)

phần nguyên 0, dư 1
   vị trí 0: chữ số 1, dư mới 4   | seen = {1: 0}
   vị trí 1: chữ số 6, dư mới 4   | seen = {1: 0, 4: 1}
   gặp lại dư 4 (lần đầu ở vị trí 1) → chu kỳ là 6
→ 0.1(6)
phần nguyên 3, dư 1
   vị trí 0: chữ số 1, dư mới 3   | seen = {1: 0}
   vị trí 1: chữ số 4, dư mới 2   | seen = {1: 0, 3: 1}
   vị trí 2: chữ số 2, dư mới 6   | seen = {1: 0, 3: 1, 2: 2}
   vị trí 3: chữ số 8, dư mới 4   | seen = {1: 0, 3: 1, 2: 2, 6: 3}
   vị trí 4: chữ số 5, dư mới 5   | seen = {1: 0, 3: 1, 2: 2, 6: 3, 4: 4}
   vị trí 5: chữ số 7, dư mới 1   | seen = {1: 0, 3: 1, 2: 2, 6: 3, 4: 4, 5: 5}
   gặp lại dư 1 (lần đầu ở vị trí 0) → chu kỳ là 142857
→ 3.(142857)


# 6. Phiên bản đúng format LeetCode

In [11]:
class Solution:
    def fractionToDecimal(self, numerator: int, denominator: int) -> str:
        if numerator == 0:
            return "0"
        out = []
        if (numerator < 0) != (denominator < 0):
            out.append("-")
        n, d = abs(numerator), abs(denominator)
        q, r = divmod(n, d)
        out.append(str(q))
        if r == 0:
            return "".join(out)
        out.append(".")
        seen = {}
        while r != 0:
            if r in seen:
                out.insert(seen[r], "(")
                out.append(")")
                break
            seen[r] = len(out)                 # vị trí trong out mà chữ số từ dư r sẽ đứng
            r *= 10
            out.append(str(r // d))
            r %= d
        return "".join(out)

Kiểm tra bằng `Fraction`: chuyển chuỗi kết quả ngược lại thành phân số (phần lặp `0.a(b)` = `a/10^k + b / (10^k · (10^m − 1))`) và so với `Fraction(tử, mẫu)`.

In [12]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<14} PASS  {actual}")


def decimal_string_to_fraction(s: str) -> Fraction:
    sign = -1 if s.startswith("-") else 1
    s = s.lstrip("-")
    if "." not in s:
        return sign * Fraction(int(s))
    ip, frac = s.split(".")
    if "(" in frac:
        pre, rep = frac.rstrip(")").split("(")
    else:
        pre, rep = frac, ""
    val = Fraction(int(ip))
    if pre:
        val += Fraction(int(pre), 10 ** len(pre))
    if rep:
        val += Fraction(int(rep), 10 ** len(pre) * (10 ** len(rep) - 1))
    return sign * val


sol = Solution()
check("1/2", sol.fractionToDecimal(1, 2), "0.5")
check("2/1", sol.fractionToDecimal(2, 1), "2")
check("4/333", sol.fractionToDecimal(4, 333), "0.(012)")
check("1/6", sol.fractionToDecimal(1, 6), "0.1(6)")
check("-50/8", sol.fractionToDecimal(-50, 8), "-6.25")
check("0/-5", sol.fractionToDecimal(0, -5), "0")
check("7/-12", sol.fractionToDecimal(7, -12), "-0.58(3)")
check("INT_MIN/-1", sol.fractionToDecimal(-2**31, -1), "2147483648")
check("-1/INT_MIN", sol.fractionToDecimal(-1, -2**31), "0.0000000004656612873077392578125")

random.seed(166)
for _ in range(5000):
    a = random.randint(-200, 200)
    b = random.choice([x for x in range(-60, 61) if x != 0])
    s = sol.fractionToDecimal(a, b)
    assert decimal_string_to_fraction(s) == Fraction(a, b), (a, b, s)
    assert s == fraction_to_decimal(a, b)
print("5000 phân số ngẫu nhiên: chuỗi kết quả đổi ngược đúng bằng phân số gốc")

print("1/97 có chu kỳ dài:", len(sol.fractionToDecimal(1, 97)), "ký tự")

1/2            PASS  0.5
2/1            PASS  2
4/333          PASS  0.(012)
1/6            PASS  0.1(6)
-50/8          PASS  -6.25
0/-5           PASS  0
7/-12          PASS  -0.58(3)
INT_MIN/-1     PASS  2147483648
-1/INT_MIN     PASS  0.0000000004656612873077392578125


5000 phân số ngẫu nhiên: chuỗi kết quả đổi ngược đúng bằng phân số gốc
1/97 có chu kỳ dài: 100 ký tự


# 7. Vì sao đúng? Độ phức tạp?

- Chữ số thứ `k` sau dấu phẩy và số dư sau bước `k` được xác định **hoàn toàn** bởi số dư trước bước đó. Vậy nếu số dư `r` xuất hiện ở bước `i` và lại ở bước `j > i`, dãy chữ số từ `i` sẽ lặp với chu kỳ `j − i` mãi mãi.
- Số dư thuộc `{1, …, d − 1}` (bỏ 0 vì 0 nghĩa là kết thúc) → sau tối đa `d − 1` bước hoặc kết thúc hoặc gặp lại → vòng lặp dừng.
- Lần đầu gặp lại là chu kỳ **ngắn nhất**, và vị trí `seen[r]` là điểm **sớm nhất** chu kỳ bắt đầu.

```text
Time:  O(L)  với L = độ dài phần thập phân (≤ d, đề đảm bảo < 10^4)
Space: O(L)  (bảng seen + chuỗi kết quả)
```

# 8. Những lỗi dễ mắc

## Lỗi 1: Dùng `float` hoặc `str(a / b)`

In [13]:
print("str(4/333) =", str(4 / 333), "← chỉ ~17 chữ số, không biết chu kỳ ở đâu")
print("str(1/97)  =", str(1 / 97), "| chu kỳ thật dài", len(sol.fractionToDecimal(1, 97)) - 4, "chữ số")

str(4/333) = 0.012012012012012012 ← chỉ ~17 chữ số, không biết chu kỳ ở đâu
str(1/97)  = 0.010309278350515464 | chu kỳ thật dài 96 chữ số


## Lỗi 2: Xử lý dấu bằng `divmod` trên số âm

In [14]:
def wrong_sign(numerator, denominator):
    q, r = divmod(numerator, denominator)          # SAI: -50 // 8 = -7 (làm tròn xuống)
    if r == 0:
        return str(q)
    digits = []
    for _ in range(4):
        r *= 10
        digits.append(str(r // denominator)); r %= denominator
        if r == 0:
            break
    return f"{q}." + "".join(digits)

print("Sai :", wrong_sign(-50, 8), "| Đúng:", sol.fractionToDecimal(-50, 8))

Sai : -7.75 | Đúng: -6.25


`divmod(-50, 8) = (-7, 6)` vì Python làm tròn **xuống**: `−50 = −7·8 + 6`. Kết quả `"-7.75"` là sai. Luôn tách dấu, làm việc với `abs`.

## Lỗi 3: Quên trường hợp tử bằng 0

In [15]:
def wrong_zero(numerator, denominator):
    sign = "-" if (numerator < 0) != (denominator < 0) else ""   # không xét tử = 0 riêng
    n, d = abs(numerator), abs(denominator)
    q, r = divmod(n, d)
    return sign + str(q) if r == 0 else "..."

print("Sai :", repr(wrong_zero(0, -5)), "| Đúng:", repr(sol.fractionToDecimal(0, -5)))

Sai : '-0' | Đúng: '0'


`0 / −5` thành `"-0"`. Cần trả `"0"` sớm khi tử bằng 0.

## Lỗi 4: Lưu **chữ số** đã gặp thay vì **số dư**

In [16]:
def wrong_seen_digit(numerator, denominator):
    n, d = abs(numerator), abs(denominator)
    q, r = divmod(n, d)
    digits, seen = [], {}
    while r != 0:
        r *= 10
        dgt = str(r // d); r %= d
        if dgt in seen:                            # SAI: cùng chữ số không có nghĩa là bắt đầu lặp
            i = seen[dgt]
            return f"{q}." + "".join(digits[:i]) + "(" + "".join(digits[i:]) + ")"
        seen[dgt] = len(digits)
        digits.append(dgt)
    return f"{q}." + "".join(digits)

print("1/7  — sai:", wrong_seen_digit(1, 7), "| đúng:", sol.fractionToDecimal(1, 7), "(tình cờ đúng)")
print("3/14 — sai:", wrong_seen_digit(3, 14), "| đúng:", sol.fractionToDecimal(3, 14))

1/7  — sai: 0.(142857) | đúng: 0.(142857) (tình cờ đúng)
3/14 — sai: 0.(214) | đúng: 0.2(142857)


`3/14 = 0.2142857142857…`: chữ số `2` xuất hiện lần hai ở vị trí 3, nhưng chu kỳ thật bắt đầu từ vị trí 1 và dài 6. Trạng thái quyết định tương lai là **số dư**, không phải chữ số vừa viết — hai chữ số giống nhau có thể đến từ hai số dư khác nhau. Với `1/7` lỗi bị che vì sáu chữ số của chu kỳ đều khác nhau.

# 9. Mẹo nhận diện pattern

- **Quá trình tất định với số trạng thái hữu hạn** (mỗi bước chỉ phụ thuộc trạng thái hiện tại) → chắc chắn rơi vào **chu kỳ**; phát hiện bằng **bảng băm** `trạng thái → thời điểm` (hoặc Floyd nhanh/chậm).
- **Tính toán chính xác trên số lớn / phân số** → số nguyên, không dùng `float`.

Bài liên quan: **#202 Happy Number** (phát hiện chu kỳ), **#142 Linked List Cycle II**, **#457 Circular Array Loop**, **#972 Equal Rational Numbers** (Hard, đổi chuỗi lặp về phân số), **#29 Divide Two Integers**.

# 10. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Chia tay `5 / 12` và `1 / 13`: liệt kê số dư từng bước, chỉ ra chỗ lặp.

## Bài 2 — Điền chỗ trống

In [17]:
def exercise_2_fraction_to_decimal(numerator: int, denominator: int) -> str:
    if numerator == 0:
        return "0"
    sign = "-" if (numerator < 0) != (denominator < 0) else ""
    n, d = abs(numerator), abs(denominator)
    q, r = divmod(n, d)
    if r == 0:
        return sign + str(q)
    digits, seen = [], {}
    # TODO: vòng lặp chia tay; khi gặp lại số dư thì chèn ngoặc
    pass


# assert exercise_2_fraction_to_decimal(1, 2) == "0.5"
# assert exercise_2_fraction_to_decimal(4, 333) == "0.(012)"
# assert exercise_2_fraction_to_decimal(1, 6) == "0.1(6)"
# assert exercise_2_fraction_to_decimal(-50, 8) == "-6.25"

## Bài 3 — Biến thể: độ dài chu kỳ của `1 / d`

Trả về độ dài phần lặp của `1/d` (`0` nếu thập phân hữu hạn), **không** tạo chuỗi.

In [18]:
def exercise_3_period_length(d: int) -> int:
    # TODO
    pass


# assert exercise_3_period_length(3) == 1
# assert exercise_3_period_length(7) == 6
# assert exercise_3_period_length(8) == 0
# assert exercise_3_period_length(12) == 1

## Bài 4 — Biến thể: chuỗi thập phân lặp → phân số tối giản

Ngược lại với bài gốc: `"0.1(6)"` → `(1, 6)`, `"-6.25"` → `(-25, 4)`. Trả về `(tử, mẫu)` tối giản, mẫu dương.

Công thức: `0.a(b)` với `a` dài `k` chữ số, `b` dài `m` chữ số bằng `(ab − a) / (10^k · (10^m − 1))` (ở đây `ab` là số ghép `a` và `b`).

In [19]:
def exercise_4_decimal_to_fraction(s: str) -> tuple:
    # TODO
    pass


# assert exercise_4_decimal_to_fraction("0.1(6)") == (1, 6)
# assert exercise_4_decimal_to_fraction("-6.25") == (-25, 4)
# assert exercise_4_decimal_to_fraction("0.(012)") == (4, 333)
# assert exercise_4_decimal_to_fraction("2") == (2, 1)

## Bài 5 — Mini challenge: chữ số thứ `k` sau dấu phẩy của `a / b` (`k` tới `10^18`)

Không thể chia tay `10^18` bước. Nhận xét: số dư sau `k − 1` bước là `a · 10^(k−1) mod b`, tính nhanh bằng `pow(10, k − 1, b)`. Giả sử `0 <= a`, `b > 0`.

In [20]:
def exercise_5_kth_digit(a: int, b: int, k: int) -> int:
    # TODO
    pass


# assert exercise_5_kth_digit(1, 7, 1) == 1 and exercise_5_kth_digit(1, 7, 2) == 4 and exercise_5_kth_digit(1, 7, 7) == 1
# assert exercise_5_kth_digit(1, 4, 3) == 0
# assert exercise_5_kth_digit(1, 7, 10**18) == int("142857"[(10**18 - 1) % 6])

# 11. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_fraction` sau khi tự làm.

**Bài 3**
- Mức 1: bỏ hết thừa số 2 và 5 khỏi `d` (chúng chỉ tạo phần **không lặp**). Còn lại `d'`; nếu `d' = 1` thì hữu hạn.
- Mức 2: độ dài chu kỳ là số `m` nhỏ nhất với `10^m ≡ 1 (mod d')`. Hoặc: chia tay và đếm từ lần gặp số dư đầu tiên tới lần gặp lại.

**Bài 4**
- Mức 1: tách phần nguyên, phần không lặp `a`, phần lặp `b`; dùng `Fraction` để cộng và tối giản.

**Bài 5**
- Mức 1: `r = a % b * pow(10, k - 1, b) % b`; chữ số là `r * 10 // b`.

# 12. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [21]:
trace_fraction(5, 12)
trace_fraction(1, 13)

phần nguyên 0, dư 5
   vị trí 0: chữ số 4, dư mới 2   | seen = {5: 0}
   vị trí 1: chữ số 1, dư mới 8   | seen = {5: 0, 2: 1}
   vị trí 2: chữ số 6, dư mới 8   | seen = {5: 0, 2: 1, 8: 2}
   gặp lại dư 8 (lần đầu ở vị trí 2) → chu kỳ là 6
→ 0.41(6)
phần nguyên 0, dư 1
   vị trí 0: chữ số 0, dư mới 10  | seen = {1: 0}
   vị trí 1: chữ số 7, dư mới 9   | seen = {1: 0, 10: 1}
   vị trí 2: chữ số 6, dư mới 12  | seen = {1: 0, 10: 1, 9: 2}
   vị trí 3: chữ số 9, dư mới 3   | seen = {1: 0, 10: 1, 9: 2, 12: 3}
   vị trí 4: chữ số 2, dư mới 4   | seen = {1: 0, 10: 1, 9: 2, 12: 3, 3: 4}
   vị trí 5: chữ số 3, dư mới 1   | seen = {1: 0, 10: 1, 9: 2, 12: 3, 3: 4, 4: 5}
   gặp lại dư 1 (lần đầu ở vị trí 0) → chu kỳ là 076923
→ 0.(076923)


## Đáp án Bài 2

In [22]:
def answer_2_fraction_to_decimal(numerator: int, denominator: int) -> str:
    return fraction_to_decimal(numerator, denominator)

assert answer_2_fraction_to_decimal(1, 2) == "0.5"
assert answer_2_fraction_to_decimal(4, 333) == "0.(012)"
assert answer_2_fraction_to_decimal(1, 6) == "0.1(6)"
assert answer_2_fraction_to_decimal(-50, 8) == "-6.25"
print("Bài 2 PASS (giống fraction_to_decimal ở mục 4)")

Bài 2 PASS (giống fraction_to_decimal ở mục 4)


## Đáp án Bài 3

In [23]:
def answer_3_period_length(d: int) -> int:
    seen, r, step = {}, 1 % d, 0
    while r != 0 and r not in seen:
        seen[r] = step
        r = r * 10 % d
        step += 1
    return 0 if r == 0 else step - seen[r]

def period_by_order(d):
    for p in (2, 5):
        while d % p == 0:
            d //= p
    if d == 1:
        return 0
    m, x = 1, 10 % d
    while x != 1:
        x = x * 10 % d; m += 1
    return m

assert answer_3_period_length(3) == 1
assert answer_3_period_length(7) == 6
assert answer_3_period_length(8) == 0
assert answer_3_period_length(12) == 1
for d in range(1, 500):
    assert answer_3_period_length(d) == period_by_order(d), d
print("Bài 3 PASS (khớp cách tính bậc của 10 modulo d')")

Bài 3 PASS (khớp cách tính bậc của 10 modulo d')


## Đáp án Bài 4

In [24]:
def answer_4_decimal_to_fraction(s: str) -> tuple:
    f = decimal_string_to_fraction(s)
    return f.numerator, f.denominator

assert answer_4_decimal_to_fraction("0.1(6)") == (1, 6)
assert answer_4_decimal_to_fraction("-6.25") == (-25, 4)
assert answer_4_decimal_to_fraction("0.(012)") == (4, 333)
assert answer_4_decimal_to_fraction("2") == (2, 1)
print("Bài 4 PASS (dùng decimal_string_to_fraction ở mục 6)")

Bài 4 PASS (dùng decimal_string_to_fraction ở mục 6)


## Đáp án Bài 5

In [25]:
def answer_5_kth_digit(a: int, b: int, k: int) -> int:
    r = (a % b) * pow(10, k - 1, b) % b        # số dư trước khi sinh chữ số thứ k
    return r * 10 // b

assert answer_5_kth_digit(1, 7, 1) == 1 and answer_5_kth_digit(1, 7, 2) == 4 and answer_5_kth_digit(1, 7, 7) == 1
assert answer_5_kth_digit(1, 4, 3) == 0
assert answer_5_kth_digit(1, 7, 10**18) == int("142857"[(10**18 - 1) % 6])
for _ in range(2000):
    a, b, k = random.randint(0, 50), random.randint(1, 50), random.randint(1, 40)
    assert answer_5_kth_digit(a, b, k) == (a * 10 ** k // b) % 10
print("Bài 5 PASS")

Bài 5 PASS


# 13. Tổng kết

> Tách dấu, làm với `abs`. Phần nguyên bằng `divmod`. Phần thập phân bằng **chia tay**: `r *= 10`, chữ số `r // d`, `r %= d`. Lưu `số dư → vị trí` trong bảng băm; gặp lại số dư nào thì chu kỳ bắt đầu từ vị trí đã lưu → chèn `(` và `)`. Số dư bằng 0 thì thập phân hữu hạn.

---

## Checklist trước khi nộp

- [ ] Tử bằng 0 → `"0"` (không `"-0"`)?
- [ ] Dấu tính riêng, chia trên `abs`?
- [ ] Bảng băm lưu **số dư**, không phải chữ số?
- [ ] Không dùng `float`?